# Imports

In [1]:
import re
import optuna
import pickle

import numpy as np
import pandas as pd

from sklearn import set_config
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier

from scipy.special import expit

from sklearn.metrics import roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import StackingClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold, train_test_split, cross_val_predict

from feature_engine.encoding import RareLabelEncoder
from feature_engine.selection import DropFeatures

/home/junior/Documentos/GitHub/prediction-f1-pit-stops-kaggle-competition/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Utils

In [2]:
set_config(enable_metadata_routing=True)

In [3]:
def dump_pickle(file_obj, file_path):
    with open(file_path, 'bw') as file:
        pickle.dump(file_obj, file)


def load_pickle(file_path):
    with open(file_path, 'rb') as file:
        return pickle.load(file)


def voting_cross_val_predict(probas: list[np.ndarray], weights: np.ndarray[float]) -> np.ndarray[float]:
    weights /= weights.sum()
    return np.sum([proba * weight for proba, weight in zip(probas, weights)], axis=0)

# Loading Dataset

In [4]:
X_train_proba = pd.read_parquet('../data/processed/X_train_stacking.parquet').rename(columns={'knn_proba': 'pca_knn_proba'})
X_test_proba = pd.read_parquet('../data/processed/X_test_stacking.parquet').rename(columns={'knn_proba': 'pca_knn_proba'})

In [5]:
X_train = pd.read_parquet('../data/processed/X_train.parquet')
X_test = pd.read_parquet('../data/processed/X_test.parquet')

y_train = pd.read_parquet('../data/processed/y_train.parquet')

In [6]:
X_train_proba.head()

,lgbm_proba,cat_proba,xgb_proba,hist_proba,lg_proba,pca_knn_proba
0,0.802607,0.891870,0.895659,0.883947,0.808497,0.887454
1,0.649405,0.831360,0.829854,0.779541,0.689888,0.519966
2,0.505904,0.851809,0.751522,0.758911,0.558520,0.209855
3,0.001713,0.005910,0.007057,0.007298,0.002275,0.000000
4,0.392205,0.565506,0.489033,0.482967,0.414144,0.297043


In [7]:
X_train.head()

,driver,compound,race,year,pitstop,lapnumber,stint,tyrelife,position,laptime_s,...,treeraceprogress,treeposition,treelaptime_s,treecumulative_degradation,treeraceprogress_position,treeraceprogress_laptime_s,treeraceprogress_cumulative_degradation,treeposition_laptime_s,treeposition_cumulative_degradation,treelaptime_s_cumulative_degradation
id,,,,,,,,,,,,,,,,,,,,,
0,D109,HARD,Canadian Grand Prix,2022,0,50,2,39.0,8,78.491,...,0.298107,0.207392,0.252153,0.279021,0.298107,0.298107,0.321361,0.252153,0.279021,0.253721
1,D086,HARD,Dutch Grand Prix,2025,1,27,2,7.0,4,75.095,...,0.298107,0.185508,0.252153,0.586119,0.298107,0.298107,0.610176,0.252153,0.544023,0.613159
2,ZON,HARD,Austrian Grand Prix,2022,0,59,3,22.0,13,70.945,...,0.298107,0.207392,0.252153,0.207676,0.298107,0.298107,0.423519,0.252153,0.228194,0.231339
3,SPE,MEDIUM,Pre-Season Testing,2023,0,2,1,2.0,7,94.361,...,0.102471,0.185508,0.178575,0.138352,0.102471,0.102471,0.043550,0.178575,0.138352,0.070991
4,D019,HARD,Azerbaijan Grand Prix,2022,1,26,3,6.0,2,107.878,...,0.298107,0.185508,0.178575,0.138352,0.298107,0.298107,0.206543,0.178575,0.138352,0.153276


# Feature Engineering

## Models

In [8]:
ridge = load_pickle('../models/model_ridge.pkl')
truncatedsvd_ridge = load_pickle('../models/model_truncatedsvd_ridge.pkl')

sgdclassifier = load_pickle('../models/model_sgdclassifier.pkl')
truncatedsvd_rbfsampler_sgdclassifier = load_pickle('../models/model_truncatedsvd_rbfsampler_sgdclassifier.pkl')
truncatedsvd_nystroem_sgdclassifier = load_pickle('../models/model_truncatedsvd_nystroem_sgdclassifier.pkl')

truncatedsvd_knn = load_pickle('../models/model_truncatedsvd_knn.pkl')
truncatedsvd_nystroem_knn = load_pickle('../models/model_truncatedsvd_nystroem_knn.pkl')
truncatedsvd_rbfsampler_knn = load_pickle('../models/model_truncatedsvd_rbfsampler_knn.pkl')

truncatedsvd_lgbm = load_pickle('../models/model_truncatedsvd_lgbm.pkl')

truncatedsvd_linear_svc = load_pickle('../models/model_truncatedsvd_linear_svc.pkl')

truncatedsvd_logistic_regression = load_pickle('../models/model_truncatedsvd_logistic_regression.pkl')
truncatedsvd_rbfsampler_logistic_regression = load_pickle('../models/model_truncatedsvd_rbfsampler_logistic_regression.pkl')
truncatedsvd_nystroem_logistic_regression = load_pickle('../models/model_truncatedsvd_nystroem_logistic_regression.pkl')

In [9]:
decision_function_dict = {
    'ridge': ridge, 
    'truncatedsvd_linear_svc': truncatedsvd_linear_svc, 
    'truncatedsvd_ridge': truncatedsvd_ridge
}

for model_name, model in decision_function_dict.items():

    print(f"using: {model_name}")

    X_train_proba[f'{model_name}_logit'] = cross_val_predict(model, X_train, y_train.PitNextLap,  cv=StratifiedKFold(shuffle=True), n_jobs=-1, method='decision_function')
    X_test_proba[f'{model_name}_logit'] = model.decision_function(X_test)

using: ridge
using: truncatedsvd_linear_svc


/home/junior/Documentos/GitHub/prediction-f1-pit-stops-kaggle-competition/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but TruncatedSVD was fitted with feature names
  warnings.warn(
/home/junior/Documentos/GitHub/prediction-f1-pit-stops-kaggle-competition/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/home/junior/Documentos/GitHub/prediction-f1-pit-stops-kaggle-competition/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LinearSVC was fitted with feature names
  warnings.warn(


using: truncatedsvd_ridge


/home/junior/Documentos/GitHub/prediction-f1-pit-stops-kaggle-competition/.venv/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/junior/Documentos/GitHub/prediction-f1-pit-stops-kaggle-competition/.venv/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/junior/Documentos/GitHub/prediction-f1-pit-stops-kaggle-competition/.venv/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/junior/Documentos/GitHub/prediction-f1-pit-stops-kaggle-competition/.venv/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/junior/Documentos/

In [10]:
proba_dict = {
    'sgdclassifier': sgdclassifier,
    'truncatedsvd_lgbm': truncatedsvd_lgbm,
    'truncatedsvd_knn': truncatedsvd_knn,
    'truncatedsvd_nystroem_knn': truncatedsvd_nystroem_knn,
    'truncatedsvd_rbfsampler_knn': truncatedsvd_rbfsampler_knn,
    'truncatedsvd_logistic_regression': truncatedsvd_logistic_regression,
    'truncatedsvd_nystroem_logistic_regression': truncatedsvd_nystroem_logistic_regression,
    'truncatedsvd_nystroem_sgdclassifier': truncatedsvd_nystroem_sgdclassifier,
    'truncatedsvd_rbfsampler_logistic_regression': truncatedsvd_rbfsampler_logistic_regression,
    'truncatedsvd_rbfsampler_sgdclassifier': truncatedsvd_rbfsampler_sgdclassifier
}


for model_name, model in proba_dict.items():

    print(f"using: {model_name}")

    X_train_proba[f'{model_name}_proba'] = cross_val_predict(model, X_train, y_train.PitNextLap,  cv=StratifiedKFold(shuffle=True), n_jobs=-1, method='predict_proba')[:, 1]
    X_test_proba[f'{model_name}_proba'] = model.predict_proba(X_test)[:, 1]

using: sgdclassifier
using: truncatedsvd_lgbm


/home/junior/Documentos/GitHub/prediction-f1-pit-stops-kaggle-competition/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/junior/Documentos/GitHub/prediction-f1-pit-stops-kaggle-competition/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/junior/Documentos/GitHub/prediction-f1-pit-stops-kaggle-competition/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/junior/Documentos/GitHub/prediction-f1-pit-stops-kaggle-competition/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted 

using: truncatedsvd_knn


/home/junior/Documentos/GitHub/prediction-f1-pit-stops-kaggle-competition/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but TruncatedSVD was fitted with feature names
  warnings.warn(
/home/junior/Documentos/GitHub/prediction-f1-pit-stops-kaggle-competition/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/home/junior/Documentos/GitHub/prediction-f1-pit-stops-kaggle-competition/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but KNeighborsClassifier was fitted with feature names
  warnings.warn(


using: truncatedsvd_nystroem_knn
using: truncatedsvd_rbfsampler_knn
using: truncatedsvd_logistic_regression


/home/junior/Documentos/GitHub/prediction-f1-pit-stops-kaggle-competition/.venv/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/junior/Documentos/GitHub/prediction-f1-pit-stops-kaggle-competition/.venv/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/junior/Documentos/GitHub/prediction-f1-pit-stops-kaggle-competition/.venv/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/junior/Documentos/GitHub/prediction-f1-pit-stops-kaggle-competition/.venv/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


using: truncatedsvd_nystroem_logistic_regression
using: truncatedsvd_nystroem_sgdclassifier
using: truncatedsvd_rbfsampler_logistic_regression
using: truncatedsvd_rbfsampler_sgdclassifier


## Voting Classifier

In [11]:
X_train_proba['voting_lgbm_and_cat_and_xgb_and_hist_proba'] = voting_cross_val_predict(
    [
        X_train_proba['lgbm_proba'],
        X_train_proba['cat_proba'],
        X_train_proba['xgb_proba'],
        X_train_proba['hist_proba']
    ], 
    weights=np.power([0.9484150827066593, 0.9481909576979181, 0.9477770044293032, 0.9485545192992528], 2)
)

X_test_proba['voting_lgbm_and_cat_and_xgb_and_hist_proba'] = voting_cross_val_predict(
    [
        X_test_proba['lgbm_proba'],
        X_test_proba['cat_proba'],
        X_test_proba['xgb_proba'],
        X_test_proba['hist_proba']
    ],
    weights=np.power([0.9484150827066593, 0.9481909576979181, 0.9477770044293032, 0.9485545192992528], 2)
)

In [12]:
column_name = (
    'voting',
    'lg',
    'ridge',
    'truncatedsvd_ridge',
    'sgdclassifier', 
    'truncatedsvd_logistic_regression',
    'truncatedsvd_nystroem_logistic_regression',
    'truncatedsvd_nystroem_sgdclassifier',
    'truncatedsvd_rbfsampler_logistic_regression',
    'truncatedsvd_rbfsampler_sgdclassifier'
)


X_train_proba['_and_'.join(column_name)] = voting_cross_val_predict(
    [
        X_train_proba['lg_proba'],
        X_train_proba['ridge_logit'].apply(expit),
        X_train_proba['truncatedsvd_ridge_logit'].apply(expit),
        X_train_proba['sgdclassifier_proba'],
        X_train_proba['truncatedsvd_logistic_regression_proba'],
        X_train_proba['truncatedsvd_nystroem_logistic_regression_proba'],
        X_train_proba['truncatedsvd_nystroem_sgdclassifier_proba'],
        X_train_proba['truncatedsvd_rbfsampler_logistic_regression_proba'],
        X_train_proba['truncatedsvd_rbfsampler_sgdclassifier_proba']
    ],
    weights=np.power(
        [
            0.8934924863693583, 
            0.8880861768599859, 
            0.8883182313760992,
            0.893129083108098,
            0.8937726214182687,
            0.8952558589759955,
            0.895307168953566,
            0.8814223922338199,
            0.8929350054167198
        ], 2)
)


X_test_proba['_and_'.join(column_name)] = voting_cross_val_predict(
    [
        X_test_proba['lg_proba'],
        X_test_proba['ridge_logit'].apply(expit),
        X_test_proba['truncatedsvd_ridge_logit'].apply(expit),
        X_test_proba['sgdclassifier_proba'],
        X_test_proba['truncatedsvd_logistic_regression_proba'],
        X_test_proba['truncatedsvd_nystroem_logistic_regression_proba'],
        X_test_proba['truncatedsvd_nystroem_sgdclassifier_proba'],
        X_test_proba['truncatedsvd_rbfsampler_logistic_regression_proba'],
        X_test_proba['truncatedsvd_rbfsampler_sgdclassifier_proba']
    ],
    weights=np.power(
        [
            0.8934924863693583, 
            0.8880861768599859, 
            0.8883182313760992,
            0.893129083108098,
            0.8937726214182687,
            0.8952558589759955,
            0.895307168953566,
            0.8814223922338199,
            0.8929350054167198
        ], 2)
)

In [13]:
X_train_proba['voting_truncatedsvd_nytroem_knn_and_truncated_rbfsampler_knn_and_truncatedsvd_knn_and_pca_knn_proba'] = voting_cross_val_predict(
    [
        X_train_proba['truncatedsvd_nystroem_knn_proba'],
        X_train_proba['truncatedsvd_rbfsampler_knn_proba'],
        X_train_proba['truncatedsvd_knn_proba'],
        X_train_proba['pca_knn_proba']
    ],
    weights=np.power([0.9013407399373964, 0.9089648111229082, 0.9111399758749241, 0.8914149276506438], 2)
)


X_test_proba['voting_truncatedsvd_nytroem_knn_and_truncated_rbfsampler_knn_and_truncatedsvd_knn_and_pca_knn_proba'] = voting_cross_val_predict(
    [
        X_test_proba['truncatedsvd_nystroem_knn_proba'],
        X_test_proba['truncatedsvd_rbfsampler_knn_proba'],
        X_test_proba['truncatedsvd_knn_proba'],
        X_test_proba['pca_knn_proba']
    ],
    weights=np.power([0.9013407399373964, 0.9089648111229082, 0.9111399758749241, 0.8914149276506438], 2)
)

In [14]:
X_test_proba.head()

,lgbm_proba,cat_proba,xgb_proba,hist_proba,lg_proba,pca_knn_proba,ridge_logit,truncatedsvd_linear_svc_logit,truncatedsvd_ridge_logit,sgdclassifier_proba,...,truncatedsvd_nystroem_knn_proba,truncatedsvd_rbfsampler_knn_proba,truncatedsvd_logistic_regression_proba,truncatedsvd_nystroem_logistic_regression_proba,truncatedsvd_nystroem_sgdclassifier_proba,truncatedsvd_rbfsampler_logistic_regression_proba,truncatedsvd_rbfsampler_sgdclassifier_proba,voting_lgbm_and_cat_and_xgb_and_hist_proba,voting_and_lg_and_ridge_and_truncatedsvd_ridge_and_sgdclassifier_and_truncatedsvd_logistic_regression_and_truncatedsvd_nystroem_logistic_regression_and_truncatedsvd_nystroem_sgdclassifier_and_truncatedsvd_rbfsampler_logistic_regression_and_truncatedsvd_rbfsampler_sgdclassifier,voting_truncatedsvd_nytroem_knn_and_truncated_rbfsampler_knn_and_truncatedsvd_knn_and_pca_knn_proba
0,0.004591,0.031986,0.022774,0.020057,0.006419,0.000000,-1.272709,-1.721325,-0.872993,0.005473,...,0.000000,0.0,0.006534,0.003712,0.009642,0.010313,0.000000,0.019849,0.061327,0.000000
1,0.003032,0.011817,0.014957,0.018887,0.063872,0.117213,-0.626960,-0.919863,-0.289991,0.054146,...,0.097699,0.1,0.063555,0.110500,0.044563,0.145672,0.047794,0.012173,0.144461,0.102869
2,0.003777,0.011641,0.014307,0.011497,0.004517,0.000000,-1.335019,-1.809246,-0.917667,0.003687,...,0.000000,0.0,0.004404,0.003123,0.008417,0.006856,0.000000,0.010304,0.057929,0.000000
3,0.163131,0.544040,0.585898,0.485388,0.655870,0.099916,0.190133,0.328363,0.387739,0.676445,...,0.000000,0.3,0.670446,0.712745,0.648495,0.504756,0.637237,0.444558,0.628242,0.183451
4,0.872172,0.967646,0.959451,0.959356,0.840450,1.000000,0.535153,0.644014,0.670147,0.852891,...,0.703383,0.8,0.835307,0.855796,0.796202,0.823601,0.787922,0.939648,0.787450,0.823541


# Machine Learning

In [16]:
def objective(trial, X, y):

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    aucs = []

    # params = {
    #     "objective": "binary",
    #     "n_estimators": trial.suggest_int("n_estimators", 30, 300),
    #     "learning_rate": trial.suggest_float("learning_rate", 0.003, 0.05, log=True),
    #     "max_depth": trial.suggest_int("max_depth", 1, 4),
    #     "num_leaves": trial.suggest_int("num_leaves", 2, 16),
    #     "min_child_samples": trial.suggest_int("min_child_samples", 20, 300),
    #     "subsample": trial.suggest_float("subsample", 0.6, 1.0),
    #     "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
    #     "reg_alpha": trial.suggest_float("reg_alpha", 1e-5, 100, log=True),
    #     "reg_lambda": trial.suggest_float("reg_lambda", 1e-5, 100, log=True),
    #     "verbosity": -1,
    # }

    for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y)):

        X_train, X_valid = X.iloc[train_idx, :], X.iloc[valid_idx, :]
        y_train, y_valid = y.iloc[train_idx, 0], y.iloc[valid_idx, 0]

        model = LGBMClassifier(
            objective='binary',
            metric='auc',
            boosting_type='gbdt',
            num_leaves=trial.suggest_int('num_leaves', 16, 256),
            max_depth=trial.suggest_int('max_depth', 3, 12),
            learning_rate=trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
            lambda_l1=trial.suggest_float('lambda_l1', 1e-3, 10.0, log=True),
            lambda_l2=trial.suggest_float('lambda_l2', 1e-3, 10.0, log=True),
            feature_fraction=trial.suggest_float('feature_fraction', 0.6, 1.0),
            bagging_fraction=trial.suggest_float('bagging_fraction', 0.6, 1.0),
            bagging_freq=trial.suggest_int('bagging_freq', 1, 7),
            min_child_samples=trial.suggest_int('min_child_samples', 10, 100),
            verbosity=-1,
            n_estimators=2000,
            random_state=42,
            n_jobs=1
        ).fit(X_train, y_train)

        proba = model.predict_proba(X_valid)[:, 1]

        auc = roc_auc_score(y_valid, proba)
        aucs.append(auc)

        trial.report(np.mean(aucs), step=fold)

        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return np.mean(aucs)

features_to_use = [
    'voting_lgbm_and_cat_and_xgb_and_hist_proba', 
    '_and_'.join(column_name), 
    'voting_truncatedsvd_nytroem_knn_and_truncated_rbfsampler_knn_and_truncatedsvd_knn_and_pca_knn_proba',
    'truncatedsvd_linear_svc_logit'
]

study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42), pruner=optuna.pruners.MedianPruner(n_warmup_steps=2))
study.optimize(lambda trial: objective(trial, X_train_proba[features_to_use], y_train), n_trials=200, n_jobs=-1, show_progress_bar=True)


print("Best trial score:")
print(study.best_trial.value)

print("\nBest params:")
print(study.best_trial.params)

[I 2026-05-24 00:14:38,652] A new study created in memory with name: no-name-829ff3fe-1764-4586-ad69-116a3756cd09
Best trial: 0. Best value: 0.949017:   0%|▌                                                                                                           | 1/200 [04:01<13:21:44, 241.73s/it]

[I 2026-05-24 00:18:40,380] Trial 0 finished with value: 0.9490166437446697 and parameters: {'num_leaves': 98, 'max_depth': 3, 'learning_rate': 0.15380286670651813, 'lambda_l1': 0.0370198118582365, 'lambda_l2': 2.005601492827561, 'feature_fraction': 0.7160773407541479, 'bagging_fraction': 0.8970142949814397, 'bagging_freq': 6, 'min_child_samples': 79}. Best is trial 0 with value: 0.9490166437446697.


Best trial: 0. Best value: 0.949017:   1%|█                                                                                                            | 2/200 [04:39<6:42:15, 121.90s/it]

[I 2026-05-24 00:19:18,392] Trial 8 finished with value: 0.9489232770633841 and parameters: {'num_leaves': 30, 'max_depth': 9, 'learning_rate': 0.045076123651945514, 'lambda_l1': 0.16174943321621266, 'lambda_l2': 0.9111062319420031, 'feature_fraction': 0.8123417369305744, 'bagging_fraction': 0.9462220838030863, 'bagging_freq': 7, 'min_child_samples': 21}. Best is trial 0 with value: 0.9490166437446697.


Best trial: 3. Best value: 0.949505:   2%|█▋                                                                                                            | 3/200 [04:49<3:51:43, 70.58s/it]

...

[I 2026-05-24 01:48:22,516] Trial 199 finished with value: 0.949883241642719 and parameters: {'num_leaves': 235, 'max_depth': 3, 'learning_rate': 0.01002655464953394, 'lambda_l1': 3.0981901558282128, 'lambda_l2': 4.686208908147776, 'feature_fraction': 0.7706938235315094, 'bagging_fraction': 0.6742770752238196, 'bagging_freq': 2, 'min_child_samples': 21}. Best is trial 186 with value: 0.9498856935231969.


Best trial: 186. Best value: 0.949886: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████▍| 199/200 [1:35:43<00:54, 54.64s/it]

[I 2026-05-24 01:50:21,813] Trial 197 pruned. 


Best trial: 186. Best value: 0.949886: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████| 200/200 [1:36:21<00:00, 28.91s/it]

[I 2026-05-24 01:51:00,641] Trial 198 finished with value: 0.9493879502002359 and parameters: {'num_leaves': 235, 'max_depth': 8, 'learning_rate': 0.010084285821589106, 'lambda_l1': 4.477919879746, 'lambda_l2': 5.691757915028568, 'feature_fraction': 0.7956218301035587, 'bagging_fraction': 0.6826343786095358, 'bagging_freq': 2, 'min_child_samples': 21}. Best is trial 186 with value: 0.9498856935231969.
Best trial score:
0.9498856935231969

Best params:
{'num_leaves': 249, 'max_depth': 3, 'learning_rate': 0.010039679039207435, 'lambda_l1': 3.778717188646178, 'lambda_l2': 3.5913533331119214, 'feature_fraction': 0.7905843441379262, 'bagging_fraction': 0.6558907986733994, 'bagging_freq': 2, 'min_child_samples': 17}


## Train Dataset

In [17]:
cv = StratifiedKFold(shuffle=True, random_state=42, n_splits=5)

stacking_proba = cross_val_predict(
    LGBMClassifier(**study.best_params), 
    X_train_proba[features_to_use], 
    y_train.PitNextLap,
    cv=cv, 
    n_jobs=-1, 
    method='predict_proba'
)[:, 1]

[LightGBM] [Warning] feature_fraction is set=0.7905843441379262, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.7905843441379262
[LightGBM] [Warning] lambda_l2 is set=3.5913533331119214, reg_lambda=0.0 will be ignored. Current value: lambda_l2=3.5913533331119214
[LightGBM] [Warning] lambda_l1 is set=3.778717188646178, reg_alpha=0.0 will be ignored. Current value: lambda_l1=3.778717188646178
[LightGBM] [Warning] bagging_fraction is set=0.6558907986733994, subsample=1.0 will be ignored. Current value: bagging_fraction=0.6558907986733994
[LightGBM] [Warning] bagging_freq is set=2, subsample_freq=0 will be ignored. Current value: bagging_freq=2
[LightGBM] [Warning] feature_fraction is set=0.7905843441379262, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.7905843441379262
[LightGBM] [Warning] lambda_l2 is set=3.5913533331119214, reg_lambda=0.0 will be ignored. Current value: lambda_l2=3.5913533331119214
[LightGBM] [Warning] lambda_l1 is set=

In [27]:
features_to_use.insert(0, 'stacking_')

In [42]:
features_to_use_rename = [
    'stacking_voting_lgbm_and_cat_and_xgb_and_hist_proba',
    'voting_and_lg_and_ridge_and_truncatedsvd_ridge_and_sgdclassifier_and_truncatedsvd_logistic_regression_and_truncatedsvd_nystroem_logistic_regression_and_truncatedsvd_nystroem_sgdclassifier_and_truncatedsvd_rbfsampler_logistic_regression_and_truncatedsvd_rbfsampler_sgdclassifier',
    'voting_truncatedsvd_nytroem_knn_and_truncated_rbfsampler_knn_and_truncatedsvd_knn_and_pca_knn_proba',
    'truncatedsvd_linear_svc_logit'
]

In [36]:
features_to_use = [
    'voting_lgbm_and_cat_and_xgb_and_hist_proba',
    'voting_and_lg_and_ridge_and_truncatedsvd_ridge_and_sgdclassifier_and_truncatedsvd_logistic_regression_and_truncatedsvd_nystroem_logistic_regression_and_truncatedsvd_nystroem_sgdclassifier_and_truncatedsvd_rbfsampler_logistic_regression_and_truncatedsvd_rbfsampler_sgdclassifier',
    'voting_truncatedsvd_nytroem_knn_and_truncated_rbfsampler_knn_and_truncatedsvd_knn_and_pca_knn_proba',
    'truncatedsvd_linear_svc_logit'
]

In [46]:
X_train_stacking = X_train_proba.copy()
X_train_stacking['_and_'.join(features_to_use_rename)] = stacking_proba

# Test Dataset

In [33]:
stacking = LGBMClassifier(**study.best_params).fit(X_train_proba[features_to_use], y_train.PitNextLap)

In [43]:
X_test_stacking = X_test_proba.copy()
X_test_stacking['_and_'.join(features_to_use_rename)] = stacking.predict_proba(X_test_proba[features_to_use])[:, 1]

# Saving

In [44]:
X_test_stacking.head()

,lgbm_proba,cat_proba,xgb_proba,hist_proba,lg_proba,pca_knn_proba,ridge_logit,truncatedsvd_linear_svc_logit,truncatedsvd_ridge_logit,sgdclassifier_proba,...,truncatedsvd_rbfsampler_knn_proba,truncatedsvd_logistic_regression_proba,truncatedsvd_nystroem_logistic_regression_proba,truncatedsvd_nystroem_sgdclassifier_proba,truncatedsvd_rbfsampler_logistic_regression_proba,truncatedsvd_rbfsampler_sgdclassifier_proba,voting_lgbm_and_cat_and_xgb_and_hist_proba,voting_and_lg_and_ridge_and_truncatedsvd_ridge_and_sgdclassifier_and_truncatedsvd_logistic_regression_and_truncatedsvd_nystroem_logistic_regression_and_truncatedsvd_nystroem_sgdclassifier_and_truncatedsvd_rbfsampler_logistic_regression_and_truncatedsvd_rbfsampler_sgdclassifier,voting_truncatedsvd_nytroem_knn_and_truncated_rbfsampler_knn_and_truncatedsvd_knn_and_pca_knn_proba,stacking_voting_lgbm_and_cat_and_xgb_and_hist_proba_and_voting_and_lg_and_ridge_and_truncatedsvd_ridge_and_sgdclassifier_and_truncatedsvd_logistic_regression_and_truncatedsvd_nystroem_logistic_regression_and_truncatedsvd_nystroem_sgdclassifier_and_truncatedsvd_rbfsampler_logistic_regression_and_truncatedsvd_rbfsampler_sgdclassifier_and_voting_truncatedsvd_nytroem_knn_and_truncated_rbfsampler_knn_and_truncatedsvd_knn_and_pca_knn_proba_and_truncatedsvd_linear_svc_logit
0,0.004591,0.031986,0.022774,0.020057,0.006419,0.000000,-1.272709,-1.721325,-0.872993,0.005473,...,0.0,0.006534,0.003712,0.009642,0.010313,0.000000,0.019849,0.061327,0.000000,0.076454
1,0.003032,0.011817,0.014957,0.018887,0.063872,0.117213,-0.626960,-0.919863,-0.289991,0.054146,...,0.1,0.063555,0.110500,0.044563,0.145672,0.047794,0.012173,0.144461,0.102869,0.089535
2,0.003777,0.011641,0.014307,0.011497,0.004517,0.000000,-1.335019,-1.809246,-0.917667,0.003687,...,0.0,0.004404,0.003123,0.008417,0.006856,0.000000,0.010304,0.057929,0.000000,0.076454
3,0.163131,0.544040,0.585898,0.485388,0.655870,0.099916,0.190133,0.328363,0.387739,0.676445,...,0.3,0.670446,0.712745,0.648495,0.504756,0.637237,0.444558,0.628242,0.183451,0.215908
4,0.872172,0.967646,0.959451,0.959356,0.840450,1.000000,0.535153,0.644014,0.670147,0.852891,...,0.8,0.835307,0.855796,0.796202,0.823601,0.787922,0.939648,0.787450,0.823541,0.643512


In [47]:
X_train_stacking.head()

,lgbm_proba,cat_proba,xgb_proba,hist_proba,lg_proba,pca_knn_proba,ridge_logit,truncatedsvd_linear_svc_logit,truncatedsvd_ridge_logit,sgdclassifier_proba,...,truncatedsvd_rbfsampler_knn_proba,truncatedsvd_logistic_regression_proba,truncatedsvd_nystroem_logistic_regression_proba,truncatedsvd_nystroem_sgdclassifier_proba,truncatedsvd_rbfsampler_logistic_regression_proba,truncatedsvd_rbfsampler_sgdclassifier_proba,voting_lgbm_and_cat_and_xgb_and_hist_proba,voting_and_lg_and_ridge_and_truncatedsvd_ridge_and_sgdclassifier_and_truncatedsvd_logistic_regression_and_truncatedsvd_nystroem_logistic_regression_and_truncatedsvd_nystroem_sgdclassifier_and_truncatedsvd_rbfsampler_logistic_regression_and_truncatedsvd_rbfsampler_sgdclassifier,voting_truncatedsvd_nytroem_knn_and_truncated_rbfsampler_knn_and_truncatedsvd_knn_and_pca_knn_proba,stacking_voting_lgbm_and_cat_and_xgb_and_hist_proba_and_voting_and_lg_and_ridge_and_truncatedsvd_ridge_and_sgdclassifier_and_truncatedsvd_logistic_regression_and_truncatedsvd_nystroem_logistic_regression_and_truncatedsvd_nystroem_sgdclassifier_and_truncatedsvd_rbfsampler_logistic_regression_and_truncatedsvd_rbfsampler_sgdclassifier_and_voting_truncatedsvd_nytroem_knn_and_truncated_rbfsampler_knn_and_truncatedsvd_knn_and_pca_knn_proba_and_truncatedsvd_linear_svc_logit
0,0.802607,0.891870,0.895659,0.883947,0.808497,0.887454,0.496500,0.527291,0.637203,0.806670,...,0.3,0.815075,0.788450,0.867595,0.911292,0.783514,0.868510,0.784140,0.618609,0.543036
1,0.649405,0.831360,0.829854,0.779541,0.689888,0.519966,0.340099,0.303278,0.512236,0.734278,...,0.7,0.686883,0.686778,0.594025,0.698730,0.695484,0.772514,0.666212,0.572044,0.446302
2,0.505904,0.851809,0.751522,0.758911,0.558520,0.209855,0.127990,0.192701,0.330117,0.533609,...,0.5,0.580053,0.607686,0.523752,0.600009,0.614942,0.717012,0.570198,0.631260,0.440385
3,0.001713,0.005910,0.007057,0.007298,0.002275,0.000000,-1.249994,-1.823374,-0.834575,0.001354,...,0.0,0.002635,0.002627,0.013206,0.025528,0.000000,0.005494,0.063219,0.000000,0.076485
4,0.392205,0.565506,0.489033,0.482967,0.414144,0.297043,-0.090382,-0.011299,0.200067,0.386963,...,0.3,0.397511,0.457883,0.364956,0.265409,0.452112,0.482416,0.418672,0.245866,0.244423


In [48]:
X_train_stacking.to_parquet('../data/processed/X_train_stacking2.parquet')
X_test_stacking.to_parquet('../data/processed/X_test_stacking2.parquet')

In [41]:
X_train_stacking.head()

,lgbm_proba,cat_proba,xgb_proba,hist_proba,lg_proba,pca_knn_proba,ridge_logit,truncatedsvd_linear_svc_logit,truncatedsvd_ridge_logit,sgdclassifier_proba,...,truncatedsvd_rbfsampler_knn_proba,truncatedsvd_logistic_regression_proba,truncatedsvd_nystroem_logistic_regression_proba,truncatedsvd_nystroem_sgdclassifier_proba,truncatedsvd_rbfsampler_logistic_regression_proba,truncatedsvd_rbfsampler_sgdclassifier_proba,voting_lgbm_and_cat_and_xgb_and_hist_proba,voting_and_lg_and_ridge_and_truncatedsvd_ridge_and_sgdclassifier_and_truncatedsvd_logistic_regression_and_truncatedsvd_nystroem_logistic_regression_and_truncatedsvd_nystroem_sgdclassifier_and_truncatedsvd_rbfsampler_logistic_regression_and_truncatedsvd_rbfsampler_sgdclassifier,voting_truncatedsvd_nytroem_knn_and_truncated_rbfsampler_knn_and_truncatedsvd_knn_and_pca_knn_proba,stacking__and_voting_lgbm_and_cat_and_xgb_and_hist_proba_and_voting_and_lg_and_ridge_and_truncatedsvd_ridge_and_sgdclassifier_and_truncatedsvd_logistic_regression_and_truncatedsvd_nystroem_logistic_regression_and_truncatedsvd_nystroem_sgdclassifier_and_truncatedsvd_rbfsampler_logistic_regression_and_truncatedsvd_rbfsampler_sgdclassifier_and_voting_truncatedsvd_nytroem_knn_and_truncated_rbfsampler_knn_and_truncatedsvd_knn_and_pca_knn_proba_and_truncatedsvd_linear_svc_logit
0,0.802607,0.891870,0.895659,0.883947,0.808497,0.887454,0.496500,0.527291,0.637203,0.806670,...,0.3,0.815075,0.788450,0.867595,0.911292,0.783514,0.868510,0.784140,0.618609,0.543036
1,0.649405,0.831360,0.829854,0.779541,0.689888,0.519966,0.340099,0.303278,0.512236,0.734278,...,0.7,0.686883,0.686778,0.594025,0.698730,0.695484,0.772514,0.666212,0.572044,0.446302
2,0.505904,0.851809,0.751522,0.758911,0.558520,0.209855,0.127990,0.192701,0.330117,0.533609,...,0.5,0.580053,0.607686,0.523752,0.600009,0.614942,0.717012,0.570198,0.631260,0.440385
3,0.001713,0.005910,0.007057,0.007298,0.002275,0.000000,-1.249994,-1.823374,-0.834575,0.001354,...,0.0,0.002635,0.002627,0.013206,0.025528,0.000000,0.005494,0.063219,0.000000,0.076485
4,0.392205,0.565506,0.489033,0.482967,0.414144,0.297043,-0.090382,-0.011299,0.200067,0.386963,...,0.3,0.397511,0.457883,0.364956,0.265409,0.452112,0.482416,0.418672,0.245866,0.244423
